# Pandas Profiling — Diagnostico automatico de un dataset

## Unidad 1: Introduccion a Desarrollo de Productos de Datos

Antes de limpiar datos, necesitamos entender que tenemos. Hacerlo columna por columna con `.info()`, `.describe()`, `.value_counts()` funciona, pero con 10 columnas y 362 filas ya se vuelve lento.

`ydata-profiling` (antes llamado `pandas-profiling`) genera un reporte HTML completo con un solo comando: tipos de datos, nulos, duplicados, distribuciones, correlaciones y alertas automaticas. En este notebook lo aplicamos al dataset `datos_feos.csv` para ver que encuentra sin que le digamos donde buscar.

### Contenido:
1. Instalacion y primer reporte
2. Interpretar las secciones del reporte
3. Configuracion y modos de ejecucion
4. Comparar dos datasets (antes vs despues de limpiar)
5. Integrar profiling en un pipeline

---
## 1. Instalacion y primer reporte

In [ ]:
# ============================================================
# INSTALACION
# ============================================================

# ydata-profiling es el nombre actual del paquete
# pandas-profiling fue el nombre hasta la version 3.x
# Ambos se importan igual: from ydata_profiling import ProfileReport

# !pip install ydata-profiling

In [ ]:
# ============================================================
# CARGAR EL DATASET
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_csv('datos_feos.csv')

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nColumnas: {df.columns.tolist()}")
df.head()

In [ ]:
# ============================================================
# GENERAR EL REPORTE COMPLETO
# ============================================================

from ydata_profiling import ProfileReport

# ProfileReport analiza el DataFrame completo
# title: titulo que aparece en el reporte HTML
# explorative: True activa analisis mas profundo (tarda mas)
profile = ProfileReport(
    df,
    title="Diagnostico — datos_feos.csv",
    explorative=True
)

# .to_notebook_iframe() muestra el reporte dentro del notebook
# Esto puede tardar 30-60 segundos dependiendo del tamano
profile.to_notebook_iframe()

In [ ]:
# ============================================================
# GUARDAR COMO HTML
# ============================================================

# El HTML se puede abrir en cualquier navegador sin Python
# Util para compartir con el equipo o con el gerente
profile.to_file("reporte_diagnostico.html")

print("Reporte guardado: reporte_diagnostico.html")

---
## 2. Interpretar las secciones del reporte

El reporte tiene varias secciones. Cada una responde una pregunta distinta sobre los datos.

### 2.1 Overview — Vision general

La primera seccion muestra:
- Numero de filas, columnas y celdas totales
- Porcentaje de celdas faltantes
- Filas duplicadas
- Tipos de variables detectados (numerica, categorica, texto, etc.)

Tambien muestra **alertas** automaticas. Estas son las mas utiles para arrancar:

In [ ]:
# ============================================================
# ACCEDER A LAS ALERTAS PROGRAMATICAMENTE
# ============================================================

# .get_description() retorna toda la info del profiling como diccionario
description = profile.get_description()

# Las alertas estan en description.alerts
print("=== ALERTAS DEL PROFILING ===")
print(f"Total de alertas: {len(description.alerts)}")
print()

for alerta in description.alerts:
    print(f"  [{alerta.alert_type.name}] {alerta.column_name}: {alerta.values}")

### 2.2 Variables — Detalle por columna

Para cada columna, el reporte muestra:
- **Tipo detectado**: numeric, categorical, text, datetime, boolean
- **Nulos**: conteo y porcentaje
- **Valores unicos**: cuantos y cuales son los mas frecuentes
- **Distribucion**: histograma para numericas, barras para categoricas
- **Estadisticas**: media, mediana, std, min, max, percentiles

Veamos como acceder a esta informacion desde Python:

In [ ]:
# ============================================================
# ESTADISTICAS POR COLUMNA
# ============================================================

# description.variables es un diccionario con info de cada columna
variables = description.variables

# Ver que info tiene disponible para una columna
col_ejemplo = "producto"
info_col = variables[col_ejemplo]

print(f"=== Columna: {col_ejemplo} ===")
print(f"  Tipo detectado: {info_col['type']}")
print(f"  Valores unicos: {info_col['n_distinct']}")
print(f"  Nulos: {info_col['n_missing']} ({info_col['p_missing']*100:.1f}%)")

In [ ]:
# ============================================================
# RESUMEN DE NULOS POR COLUMNA
# ============================================================

# Construir un resumen rapido de nulos usando los datos del profiling
resumen_nulos = []
for col_name, col_info in variables.items():
    n_missing = col_info.get('n_missing', 0)
    p_missing = col_info.get('p_missing', 0)
    tipo = col_info.get('type', 'Unknown')
    resumen_nulos.append({
        'columna': col_name,
        'tipo_detectado': str(tipo),
        'nulos': n_missing,
        'porcentaje': round(p_missing * 100, 1),
    })

df_nulos = pd.DataFrame(resumen_nulos).sort_values('nulos', ascending=False)
df_nulos

### 2.3 Correlaciones

El reporte calcula correlaciones entre variables numericas automaticamente. En nuestro dataset la mayoria de columnas son texto, asi que esta seccion sera limitada. Pero cuando se usa con datos numericos limpios, detecta relaciones fuertes y posible multicolinealidad.

Los tipos de correlacion que calcula:
- **Pearson**: relacion lineal entre dos variables
- **Spearman**: relacion monotonica (no necesariamente lineal)
- **Phik**: correlacion para variables mixtas (numericas y categoricas)

### 2.4 Missing values — Patrones de nulos

Esta seccion es particularmente util. No solo muestra cuantos nulos hay, sino **donde estan**: muestra un mapa de calor donde cada fila es un registro y cada columna es una variable. Si los nulos forman bloques o patrones, es senal de un problema sistematico (una fuente que no envia ciertos campos, un periodo sin datos, etc.).

### 2.5 Duplicados

El reporte detecta filas duplicadas exactas y muestra cuantas y cuales son. En nuestro dataset introdujimos 12 duplicados intencionalmente.

In [ ]:
# ============================================================
# DUPLICADOS DESDE EL PROFILING
# ============================================================

n_dup = description.duplicates
print(f"Filas duplicadas detectadas: {n_dup}")
print(f"Porcentaje: {n_dup / len(df) * 100:.1f}%")

---
## 3. Configuracion y modos de ejecucion

El reporte completo puede tardar varios minutos con datasets grandes. Hay formas de hacerlo mas rapido o mas enfocado.

In [ ]:
# ============================================================
# MODO MINIMAL — rapido, sin correlaciones ni interacciones
# ============================================================

# Ideal para datasets grandes (>100K filas) o para un primer vistazo
profile_minimal = ProfileReport(
    df,
    title="Diagnostico Rapido",
    minimal=True   # Solo estadisticas basicas, sin graficos pesados
)

# Este se genera mucho mas rapido
profile_minimal.to_notebook_iframe()

In [ ]:
# ============================================================
# CONFIGURACION PERSONALIZADA
# ============================================================

# Se puede controlar exactamente que calcular y que no
profile_custom = ProfileReport(
    df,
    title="Diagnostico Personalizado",
    
    # Activar/desactivar secciones
    correlations={
        "pearson": {"calculate": True},
        "spearman": {"calculate": False},  # Desactivar Spearman
        "phik": {"calculate": False},       # Desactivar Phik (lento)
    },
    
    # Umbrales de alertas
    missing_diagrams={
        "bar": True,       # Grafico de barras de nulos
        "matrix": True,    # Matriz de nulos (patron)
        "heatmap": False,  # Correlacion de nulos
    },
    
    # Muestreo para datasets grandes
    # samples={"head": 10, "tail": 10},
    
    # Interacciones entre variables
    interactions={"continuous": False},  # Desactivar scatter plots
)

profile_custom.to_file("reporte_custom.html")
print("Reporte personalizado guardado")

In [ ]:
# ============================================================
# PROFILING DE UN SUBCONJUNTO DE COLUMNAS
# ============================================================

# A veces solo interesa analizar ciertas columnas
cols_interes = ['producto', 'region', 'unidades', 'precio_unitario']

profile_parcial = ProfileReport(
    df[cols_interes],
    title="Diagnostico — Solo columnas criticas",
    minimal=True
)

profile_parcial.to_notebook_iframe()

In [ ]:
# ============================================================
# PROFILING CON DATASET GRANDE — SAMPLING
# ============================================================

# Con datasets de millones de filas, conviene muestrear
# El profiling sobre la muestra da una idea general sin esperar horas

# Ejemplo: tomar 10,000 filas aleatorias
# df_muestra = df.sample(n=10000, random_state=42)
# profile_muestra = ProfileReport(df_muestra, title="Muestra 10K", minimal=True)

print("Para datasets grandes:")
print("  1. Usar minimal=True")
print("  2. Muestrear con df.sample()")
print("  3. Seleccionar solo las columnas que interesan")
print("  4. Desactivar correlaciones y interacciones")

---
## 4. Comparar dos datasets — antes vs despues

Una de las funciones mas utiles de profiling es comparar dos versiones del mismo dataset. Permite verificar que la limpieza realmente mejoro los datos.

In [ ]:
# ============================================================
# LIMPIAR UNA VERSION RAPIDA PARA COMPARAR
# ============================================================

# Hacemos una limpieza basica para tener un "despues" que comparar
# (la limpieza completa es el ejercicio aparte)

df_limpio = df.copy()

# Eliminar duplicados
df_limpio = df_limpio.drop_duplicates()

# Producto: limpiar basico
df_limpio['producto'] = (df_limpio['producto']
    .str.replace(r"[\[\]']", "", regex=True)  # quitar corchetes y comillas
    .str.strip()                                # quitar espacios
    .str.lower()                                # minusculas
    .replace({'n/a': np.nan, '-': np.nan, '': np.nan})
)

# Region: limpiar basico
df_limpio['region'] = (df_limpio['region']
    .str.strip()
    .str.lower()
    .replace({'': np.nan, ' ': np.nan, 'desconocida': np.nan})
)

# Unidades: extraer numeros
df_limpio['unidades'] = (df_limpio['unidades']
    .str.replace(r'[^\d.\-]', '', regex=True)  # solo numeros
    .replace('', np.nan)
    .astype(float)
)

# Precio: extraer numeros
df_limpio['precio_unitario'] = (df_limpio['precio_unitario']
    .str.replace(r'[^\d.\-]', '', regex=True)
    .replace('', np.nan)
    .astype(float)
)

print(f"Antes: {len(df)} filas")
print(f"Despues: {len(df_limpio)} filas")
print(f"Producto valores unicos: {df_limpio['producto'].nunique()}")
print(f"Region valores unicos: {df_limpio['region'].nunique()}")

In [ ]:
# ============================================================
# GENERAR REPORTE COMPARATIVO
# ============================================================

# Crear profiles para ambas versiones
profile_antes = ProfileReport(
    df,
    title="ANTES — datos_feos.csv",
    minimal=True
)

profile_despues = ProfileReport(
    df_limpio,
    title="DESPUES — limpieza basica",
    minimal=True
)

# .compare() genera un reporte lado a lado
comparacion = profile_antes.compare(profile_despues)

# Mostrar en el notebook
comparacion.to_notebook_iframe()

In [ ]:
# ============================================================
# GUARDAR LA COMPARACION COMO HTML
# ============================================================

comparacion.to_file("reporte_comparativo.html")
print("Reporte comparativo guardado: reporte_comparativo.html")
print("Abrir en el navegador para ver las diferencias lado a lado.")

---
## 5. Integrar profiling en un pipeline

El profiling no es solo para exploracion interactiva. Se puede integrar como un paso automatico en un pipeline de datos para generar reportes de calidad cada vez que llegan datos nuevos.

In [ ]:
# ============================================================
# FUNCION DE DIAGNOSTICO AUTOMATICO
# ============================================================

import os
from datetime import datetime

def diagnosticar_dataset(
    df,
    nombre,
    carpeta_reportes="reportes_calidad",
    modo="minimal"
):
    """
    Genera un reporte de profiling y lo guarda con timestamp.
    Pensado para correr automaticamente cada vez que llegan datos nuevos.
    
    Parametros:
        df (DataFrame): datos a diagnosticar
        nombre (str): nombre del dataset
        carpeta_reportes (str): donde guardar los HTML
        modo (str): 'minimal' para rapido, 'full' para completo
    
    Retorna:
        dict: resumen con metricas clave
    """
    os.makedirs(carpeta_reportes, exist_ok=True)
    
    # Generar el profile
    profile = ProfileReport(
        df,
        title=f"Diagnostico — {nombre}",
        minimal=(modo == "minimal")
    )
    
    # Guardar con timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    ruta = os.path.join(
        carpeta_reportes,
        f"diagnostico_{nombre}_{timestamp}.html"
    )
    profile.to_file(ruta)
    
    # Extraer metricas clave
    desc = profile.get_description()
    
    resumen = {
        "nombre": nombre,
        "timestamp": timestamp,
        "filas": len(df),
        "columnas": len(df.columns),
        "duplicados": desc.duplicates,
        "nulos_totales": df.isnull().sum().sum(),
        "porcentaje_nulos": round(df.isnull().sum().sum() / df.size * 100, 2),
        "alertas": len(desc.alerts),
        "reporte": ruta,
    }
    
    print(f"Diagnostico generado: {ruta}")
    print(f"  Filas: {resumen['filas']}")
    print(f"  Duplicados: {resumen['duplicados']}")
    print(f"  Nulos: {resumen['nulos_totales']} ({resumen['porcentaje_nulos']}%)")
    print(f"  Alertas: {resumen['alertas']}")
    
    return resumen

In [ ]:
# ============================================================
# EJECUTAR EL DIAGNOSTICO
# ============================================================

# Sobre los datos crudos
resumen_crudo = diagnosticar_dataset(df, "datos_feos")

print()

# Sobre los datos limpios
resumen_limpio = diagnosticar_dataset(df_limpio, "datos_limpios")

In [ ]:
# ============================================================
# TABLA COMPARATIVA DE METRICAS
# ============================================================

comparativa = pd.DataFrame([resumen_crudo, resumen_limpio])
comparativa = comparativa[['nombre', 'filas', 'duplicados', 'nulos_totales', 'porcentaje_nulos', 'alertas']]
comparativa

In [ ]:
# ============================================================
# EXTRAER DATOS DEL PROFILING COMO JSON
# ============================================================

# Util para almacenar metricas de calidad en una base de datos
# y hacer seguimiento historico

profile_json = ProfileReport(
    df,
    title="Para JSON",
    minimal=True
)

# Guardar como JSON
profile_json.to_file("reporte_diagnostico.json")

print("Reporte JSON guardado: reporte_diagnostico.json")
print("Util para cargar en dashboards o bases de datos de metricas.")

---
## 6. Alternativas a ydata-profiling

ydata-profiling es el mas completo, pero hay alternativas mas ligeras segun el caso:

In [ ]:
# ============================================================
# SWEETVIZ — mas rapido, orientado a comparacion
# ============================================================

# !pip install sweetviz

# import sweetviz as sv
# report = sv.analyze(df)
# report.show_html("sweetviz_report.html")

# Comparar dos datasets:
# report = sv.compare([df, "Antes"], [df_limpio, "Despues"])
# report.show_html("sweetviz_comparacion.html")

print("sweetviz: rapido, bueno para comparar antes vs despues")

In [ ]:
# ============================================================
# SKIMPY — resumen en consola, sin HTML
# ============================================================

# !pip install skimpy

# from skimpy import skim
# skim(df)

# Imprime un resumen tipo R::skimr directamente en la terminal
# No genera HTML, solo texto formateado
# Ideal para scripts y pipelines donde no necesitas un archivo

print("skimpy: resumen rapido en consola, sin generar archivos")

In [ ]:
# ============================================================
# DATAPREP — profiling + limpieza en un solo paquete
# ============================================================

# !pip install dataprep

# from dataprep.eda import create_report
# report = create_report(df, title="DataPrep Report")
# report.save("dataprep_report.html")

# DataPrep tambien tiene funciones de limpieza:
# from dataprep.clean import clean_email, clean_phone

print("dataprep: profiling + limpieza integrada")

### Cuando usar cada herramienta

| Herramienta | Velocidad | Output | Mejor para |
|---|---|---|---|
| **ydata-profiling** | Lento | HTML completo | Diagnostico exhaustivo, compartir con equipo |
| **sweetviz** | Medio | HTML limpio | Comparar antes vs despues |
| **skimpy** | Rapido | Consola | Vistazo rapido en terminal o script |
| **dataprep** | Medio | HTML | Profiling + limpieza en uno |

---
## Resumen

| Concepto | Lo esencial |
|---|---|
| **ProfileReport(df)** | Genera el reporte completo con un comando |
| **.to_notebook_iframe()** | Muestra el reporte dentro del notebook |
| **.to_file()** | Guarda como HTML o JSON |
| **minimal=True** | Modo rapido para datasets grandes |
| **.compare()** | Comparar dos versiones del dataset lado a lado |
| **.get_description()** | Acceder a metricas y alertas desde Python |
| **Alertas** | Lo mas util: el profiling te dice donde estan los problemas antes de que los busques |

### Flujo recomendado

1. Cargar datos crudos
2. Correr profiling (minimal si es grande)
3. Leer las alertas — ahi esta el 80% del diagnostico
4. Limpiar
5. Correr profiling de nuevo y comparar
6. Si las alertas bajan y los tipos son correctos, los datos estan listos